# W08 — Assignment (SQL limpieza + Many-to-Many)

**Único entregable semanal.**

## Setup

In [1]:
from pathlib import Path
import duckdb

PROJECT_ROOT = Path(".").resolve()
RAW_CSV = PROJECT_ROOT / "data" / "raw" / "pscomppars.csv"
DB_PATH = PROJECT_ROOT / "data" / "exoplanets.duckdb"

if not RAW_CSV.exists():
    raise FileNotFoundError(f"Missing {RAW_CSV}. Run W01/W02 download first.")

def sql_path(p: Path) -> str:
    return "'" + p.resolve().as_posix().replace("'", "''") + "'"

con = duckdb.connect(str(DB_PATH))

con.execute("DROP VIEW IF EXISTS raw_ps")
con.execute(f"CREATE VIEW raw_ps AS SELECT * FROM read_csv_auto({sql_path(RAW_CSV)})")

con.sql("SELECT COUNT(*) AS n_raw FROM raw_ps").show()

┌───────┐
│ n_raw │
│ int64 │
├───────┤
│  6291 │
└───────┘



## Parte A — Limpieza Raw→Silver v2

In [2]:
con.sql("SELECT discoverymethod, COUNT(*) AS n FROM raw_ps WHERE discoverymethod IS NOT NULL GROUP BY discoverymethod ORDER BY n DESC LIMIT 15").show()
con.sql("SELECT COUNT(DISTINCT discoverymethod) AS n_unique_methods FROM raw_ps WHERE discoverymethod IS NOT NULL").show()

┌───────────────────────────────┬───────┐
│        discoverymethod        │   n   │
│            varchar            │ int64 │
├───────────────────────────────┼───────┤
│ Transit                       │  4651 │
│ Radial Velocity               │  1181 │
│ Microlensing                  │   278 │
│ Imaging                       │    97 │
│ Transit Timing Variations     │    41 │
│ Eclipse Timing Variations     │    17 │
│ Orbital Brightness Modulation │     9 │
│ Pulsar Timing                 │     8 │
│ Astrometry                    │     6 │
│ Pulsation Timing Variations   │     2 │
│ Disk Kinematics               │     1 │
└───────────────────────────────┴───────┘
  11 rows                     2 columns

┌──────────────────┐
│ n_unique_methods │
│      int64       │
├──────────────────┤
│               11 │
└──────────────────┘



### TODO A1 — method_map

In [4]:
# TODO A1: crea una tabla method_map con mapeos raw -> canonical (>=6).
con.execute("DROP TABLE IF EXISTS method_map")
con.execute("CREATE TABLE method_map(raw_method VARCHAR, canonical_method VARCHAR)")
con.execute("""
INSERT INTO method_map VALUES
  ('Transit', 'transit'), ('transit', 'transit'), ('Radial Velocity', 'radial_velocity'),
  ('radial velocity', 'radial_velocity'), ('Imaging', 'imaging'),
  ('Microlensing', 'microlensing'), ('Timing', 'timing')
""")
con.sql("SELECT * FROM method_map").show()

┌─────────────────┬──────────────────┐
│   raw_method    │ canonical_method │
│     varchar     │     varchar      │
├─────────────────┼──────────────────┤
│ Transit         │ transit          │
│ transit         │ transit          │
│ Radial Velocity │ radial_velocity  │
│ radial velocity │ radial_velocity  │
│ Imaging         │ imaging          │
│ Microlensing    │ microlensing     │
│ Timing          │ timing           │
└─────────────────┴──────────────────┘



### TODO A2 — silver_planet_v2

In [6]:
con.execute("DROP TABLE IF EXISTS method_map")
con.execute("CREATE TABLE method_map(raw_method VARCHAR, canonical_method VARCHAR)")
con.execute("""
INSERT INTO method_map VALUES
  ('Transit', 'transit'), ('transit', 'transit'), ('Radial Velocity', 'radial_velocity'),
  ('radial velocity', 'radial_velocity'), ('Imaging', 'imaging'),
  ('Microlensing', 'microlensing'), ('Timing', 'timing')
""")
con.sql("SELECT * FROM method_map").show()

# A2: silver_planet_v2
con.execute("DROP TABLE IF EXISTS silver_planet_v2")
con.execute("""
CREATE TABLE silver_planet_v2 AS
WITH base AS (
  SELECT
    pl_name, hostname, discoverymethod, disc_year,
    LOWER(TRIM(hostname)) AS hostname_clean,
    NULLIF(TRIM(discoverymethod), '') AS discoverymethod_norm
  FROM raw_ps
  WHERE pl_name IS NOT NULL AND hostname IS NOT NULL
),
mapped AS (
  SELECT b.*,
    COALESCE(m.canonical_method, LOWER(TRIM(b.discoverymethod_norm))) AS discoverymethod_clean,
    CASE
      WHEN b.disc_year IS NULL THEN NULL
      WHEN b.disc_year < 1990 THEN 'pre_1990'
      WHEN b.disc_year < 2000 THEN '1990s'
      WHEN b.disc_year < 2010 THEN '2000s'
      WHEN b.disc_year < 2020 THEN '2010s'
      ELSE '2020s'
    END AS disc_era
  FROM base b
  LEFT JOIN method_map m ON m.raw_method = b.discoverymethod
)
SELECT * FROM mapped
""")
print("Filas en silver_planet_v2:", con.execute("SELECT COUNT(*) FROM silver_planet_v2").fetchone()[0])

┌─────────────────┬──────────────────┐
│   raw_method    │ canonical_method │
│     varchar     │     varchar      │
├─────────────────┼──────────────────┤
│ Transit         │ transit          │
│ transit         │ transit          │
│ Radial Velocity │ radial_velocity  │
│ radial velocity │ radial_velocity  │
│ Imaging         │ imaging          │
│ Microlensing    │ microlensing     │
│ Timing          │ timing           │
└─────────────────┴──────────────────┘

Filas en silver_planet_v2: 6291


## Parte B — Many-to-Many (toy schema)

In [7]:
# TODO B1: construye un ejemplo M:N (toy schema) y responde 2 preguntas.
#
# REQUISITO (para que cuente como completo):
# - Debes crear el esquema con **link table** + **PK/FK**:
#   - planet_demo(planet_id PRIMARY KEY, name NOT NULL)
#   - method_demo(method_id PRIMARY KEY, method_name UNIQUE NOT NULL)
#   - planet_method_demo(planet_id, method_id) con:
#       PRIMARY KEY (planet_id, method_id)
#       FOREIGN KEY (planet_id) REFERENCES planet_demo(planet_id)
#       FOREIGN KEY (method_id) REFERENCES method_demo(method_id)
#
# - Inserta al menos 4 planetas, 3 métodos y relaciones M:N (un planeta con 2 métodos).
#
# Nota idempotencia (FK-safe): dropea puente primero.
con.execute("DROP TABLE IF EXISTS planet_method_demo")
con.execute("DROP TABLE IF EXISTS method_demo")
con.execute("DROP TABLE IF EXISTS planet_demo")

con.execute("CREATE TABLE planet_demo(planet_id INTEGER PRIMARY KEY, name VARCHAR NOT NULL)")
con.execute("CREATE TABLE method_demo(method_id INTEGER PRIMARY KEY, method_name VARCHAR NOT NULL UNIQUE)")
con.execute("""
CREATE TABLE planet_method_demo(
  planet_id INTEGER NOT NULL,
  method_id INTEGER NOT NULL,
  PRIMARY KEY (planet_id, method_id),
  FOREIGN KEY (planet_id) REFERENCES planet_demo(planet_id),
  FOREIGN KEY (method_id) REFERENCES method_demo(method_id)
)
""")

con.execute("INSERT INTO planet_demo VALUES (1,'Aurelia'), (2,'Borealis'), (3,'Cetus'), (4,'Draco')")
con.execute("INSERT INTO method_demo VALUES (10,'transit'), (20,'radial_velocity'), (30,'imaging')")
con.execute("""
INSERT INTO planet_method_demo VALUES
  (1,10), (1,20), (2,10), (3,30), (4,20), (4,30)
""")

# Q1: planetas por método
q1 = """
SELECT m.method_name, COUNT(DISTINCT pm.planet_id) AS n_planets
FROM method_demo m
JOIN planet_method_demo pm ON pm.method_id = m.method_id
GROUP BY m.method_name
"""
con.sql(q1).show()

# Q2: métodos por planeta
q2 = """
SELECT p.name, COUNT(DISTINCT pm.method_id) AS n_methods
FROM planet_demo p
JOIN planet_method_demo pm ON pm.planet_id = p.planet_id
GROUP BY p.name
"""
con.sql(q2).show()

# Verificar que la PK compuesta evita duplicados
dup_check = con.execute("""
SELECT planet_id, method_id, COUNT(*)
FROM planet_method_demo
GROUP BY planet_id, method_id
HAVING COUNT(*) > 1
""").fetchall()
print(f"Duplicados en link table: {len(dup_check)} (debe ser 0)")

con.close()

┌─────────────────┬───────────┐
│   method_name   │ n_planets │
│     varchar     │   int64   │
├─────────────────┼───────────┤
│ imaging         │         2 │
│ radial_velocity │         2 │
│ transit         │         2 │
└─────────────────┴───────────┘

┌──────────┬───────────┐
│   name   │ n_methods │
│ varchar  │   int64   │
├──────────┼───────────┤
│ Aurelia  │         2 │
│ Cetus    │         1 │
│ Draco    │         2 │
│ Borealis │         1 │
└──────────┴───────────┘

Duplicados en link table: 0 (debe ser 0)


## Entregable único semanal (W08)
- Ejecuta el assignment.
- Entrega:
  1) `docs/w08_report.md` (copiar template)
  2) 1 entrada nueva en `docs/decisions_log.md` (copiar template)

**Extra requerido (M:N):** incluye evidencia de PK/FK en tu DDL y/o el check `HAVING COUNT(*)>1` retornando vacío.